# FAST-UAV - Design of Experiments 

*Author: Félix Pollet - 2023* <br>
doe_fast` 모듈은 FAST-UAV 문제에 대한 실험 설계를 실행할 수 있는 기능을 제공합니다.

다양한 제너레이터를 사용할 수 있습니다:
* 'List' 생성기는 사용자 정의된 DoE 사례 배열에서 사례를 읽습니다.
* 균일` 생성기는 균일 분포에서 추출한 무작위 샘플을 생성합니다.
* 라틴 하이퍼큐브` 생성기는 [라틴 하이퍼큐브 샘플링](https://en.wikipedia.org/wiki/Latin_hypercube_sampling)을 인스턴스화합니다.
* Sobol` 생성기는 [SALib에서 제공하는 Sobol-Saltelli 2002 방법](https://salib.readthedocs.io/en/latest/api.html#sobol-sensitivity-analysis)을 인스턴스화합니다. 분산 기반 민감도 분석](https://en.wikipedia.org/wiki/Variance-based_sensitivity_analysis)의 한 형태인 소볼 분석을 수행하는 데 유용합니다.
* 모리스` 생성기는 [SALib에서 제공하는 모리스의 방법](https://salib.readthedocs.io/en/latest/api.html#method-of-morris)을 인스턴스화합니다. 모리스의 방법](https://en.wikipedia.org/wiki/Morris_method)은 민감도 분석을 위한 한 번에 한 단계씩 수행하는 방법(OAT)입니다.

아래 예시는 이 모듈의 몇 가지 적용 사례를 보여줍니다.

In [8]:
# Import required librairies
import os.path as pth
import fastoad.api as oad
from fastuav.utils.postprocessing.sensitivity_analysis.sensitivity_analysis import doe_fast
import numpy as np
import plotly.graph_objects as go
import plotly

# Declare paths to folders and files
DATA_FOLDER_PATH = "./data"
WORK_FOLDER_PATH = r"d:\KangDH\fast_UAV\FAST-UAV\src\fastuav\notebooks\workdir"
#WORK_FOLDER_PATH = "./workdir"
CONFIGURATION_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "configurations")
SOURCE_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "source_files")

In [18]:
# From MathWorks repository (optional external data source)
import re

mathworks_repo_path = r"D:\KangDH\fast_UAV\Quadcopter-Drone-Model-Simscape"
mathworks_data_path = pth.join(mathworks_repo_path, "Scripts_Data")
mathworks_param_file = pth.join(mathworks_data_path, "quadcopter_package_parameters.m")


def _read_scalar_from_m(file_path, var_name):
    """Read scalar assignment like `var = 1.23;` or `var = 7.6*3;` from a .m file."""
    if not pth.exists(file_path):
        return None

    text = open(file_path, "r", encoding="utf-8", errors="ignore").read()
    match = re.search(rf"^\s*{re.escape(var_name)}\s*=\s*([^;]+);", text, re.MULTILINE)
    if not match:
        return None

    expr = match.group(1).strip()
    if not re.match(r"^[0-9eE+\-*/().\s]+$", expr):
        return None

    try:
        return float(eval(expr, {"__builtins__": {}}, {}))
    except Exception:
        return None


def _replace_value(xml_text, pattern, value):
    return re.sub(pattern, lambda m: f"{m.group(1)}{value}{m.group(3)}", xml_text, count=1)


SOURCE_FILE_DOE_OPTIM = pth.join(SOURCE_FOLDER_PATH, "problem_inputs_doe_optim.xml")
MATHWORKS_DOE_SOURCE_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_inputs_doe_optim_mathworks_quad.xml")

if pth.exists(mathworks_data_path):
    print("MathWorks data folder found:", mathworks_data_path)
else:
    print("MathWorks data folder not found:", mathworks_data_path)

if pth.exists(mathworks_param_file) and pth.exists(SOURCE_FILE_DOE_OPTIM):
    print("MathWorks parameter file found:", mathworks_param_file)

    prop_diameter = _read_scalar_from_m(mathworks_param_file, "propeller.diameter")
    motor_torque_max = _read_scalar_from_m(mathworks_param_file, "qc_motor.max_torque")
    drone_mass = _read_scalar_from_m(mathworks_param_file, "drone_mass")

    xml = open(SOURCE_FILE_DOE_OPTIM, "r", encoding="utf-8", errors="ignore").read()

    # Force quadcopter topology
    xml = _replace_value(
        xml,
        r"(<number is_input=\"True\">)([^<]+)(<!--Number of propellers for multirotor-->)",
        "4.0",
    )

    # Align mission payload with MathWorks package-delivery scale (small quad)
    xml = _replace_value(
        xml,
        r"(<mass units=\"kg\" is_input=\"True\">)([^<]+)(<!--Design payload mass-->)",
        "1.0",
    )

    if prop_diameter is not None:
        xml = _replace_value(
            xml,
            r"(<reference units=\"m\" is_input=\"True\">)([^<]+)(<!--Multirotor propeller diameter reference for scaling law-->)",
            f"{prop_diameter:.6f}",
        )

    if motor_torque_max is not None:
        xml = _replace_value(
            xml,
            r"(<reference units=\"N\*m\" is_input=\"True\">)([^<]+)(<!--Multirotor motor max\. torque reference for scaling law-->)",
            f"{motor_torque_max:.6f}",
        )

    if drone_mass is not None:
        xml = _replace_value(
            xml,
            r"(<reference units=\"kg\" is_input=\"True\">)([^<]+)(<!--MTOW reference for scaling law-->)",
            f"{drone_mass:.6f}",
        )

    with open(MATHWORKS_DOE_SOURCE_FILE, "w", encoding="utf-8") as f:
        f.write(xml)

    print("Generated DoE source from MathWorks parameters:", MATHWORKS_DOE_SOURCE_FILE)
    print("Applied values -> N_prop=4.0, payload=1.0kg, D_prop_ref=", prop_diameter, ", T_max_ref=", motor_torque_max, ", MTOW_ref=", drone_mass)
else:
    print("MathWorks parameter file or base DoE source file missing.")
    print("Fallback source file will be used:", SOURCE_FILE_DOE_OPTIM)

MathWorks data folder found: D:\KangDH\fast_UAV\Quadcopter-Drone-Model-Simscape\Scripts_Data
MathWorks parameter file found: D:\KangDH\fast_UAV\Quadcopter-Drone-Model-Simscape\Scripts_Data\quadcopter_package_parameters.m
Generated DoE source from MathWorks parameters: ./data\source_files\problem_inputs_doe_optim_mathworks_quad.xml
Applied values -> N_prop=4.0, payload=1.0kg, D_prop_ref= 0.254 , T_max_ref= 0.8 , MTOW_ref= 1.2726


## Example 1: running a model for many different input values

In [3]:
CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "doe_simple_model.yaml")
SOURCE_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_inputs_doe.xml")
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)

'D:/KangDH/fast_UAV/FAST-UAV/src/fastuav/notebooks/workdir/problem_inputs.xml'

In [4]:
INPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_inputs.xml")
oad.variable_viewer(INPUT_FILE)

In [5]:
# Definition of the DoE

# Inputs to be varied
x_dict = {
    "mission:operational:main_route:payload:mass": [0.5, 4.0],  # range of variation
    "mission:operational:main_route:cruise:speed": [10.0, 20.0]
         }

# Output(s) to observe
y_list = [
    "mission:operational:main_route:energy",
]

In [6]:
# Run DoE
method = "fullfactorial"  # full factorial sampling
n_samples = 10  # Number of samples
df = doe_fast(method, x_dict, y_list, CONFIGURATION_FILE, n_samples)

c:\Users\user\.conda\envs\FastUAV_edit\lib\site-packages\openmdao\visualization\n2_viewer\n2_viewer.py:168: OpenMDAOWarning:All-NaN slice encountered
c:\Users\user\.conda\envs\FastUAV_edit\lib\site-packages\openmdao\visualization\n2_viewer\n2_viewer.py:168: OpenMDAOWarning:All-NaN slice encountered


In [7]:
# Plot DoE with Plotly

# Set variables to plot
x_name = list(x_dict.keys())[0]
y_name = list(x_dict.keys())[1]
z_name = y_list[0]

# Reshape data into a grid
x_vals = sorted(df[x_name].unique())
y_vals = sorted(df[y_name].unique())
#y_vals = [val / 1000 for val in y_vals]
z_grid = df.pivot(index=y_name, columns=x_name, values=z_name).values

# Create the contour plot    
fig = go.Figure()
fig.add_trace(go.Contour(
    x=x_vals, y=y_vals, z=z_grid, 
    connectgaps=True,
    contours=dict(start=df[z_name].min(), end=df[z_name].max(), showlabels=True),
    colorscale='RdBu_r' # Blackbody,Bluered,Blues,Cividis,Earth,Electric,Greens,Greys,Hot,Jet,Picnic,Portland,Rainbow,RdBu,Reds,Viridis,YlGnBu,YlOrRd. (add '_r' for reversed)
))

# Display the plot
fig.show()

## Example 2: DoE on the optimization problem

Sometimes you may want to run a DoE on an optimization problem rather than a model. The `doe_fast` module automatically detects if an optimization problem is defined in the configuration file, and each point of the DoE will run an optimization.

In [19]:
CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "doe_optimization_problem.yaml")

# Use MathWorks-derived quad source when available, otherwise fallback to default
if 'MATHWORKS_DOE_SOURCE_FILE' in globals() and pth.exists(MATHWORKS_DOE_SOURCE_FILE):
    SOURCE_FILE = MATHWORKS_DOE_SOURCE_FILE
else:
    SOURCE_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_inputs_doe_optim.xml")

print("DoE optimization source:", SOURCE_FILE)
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)

DoE optimization source: ./data\source_files\problem_inputs_doe_optim_mathworks_quad.xml


'D:/KangDH/fast_UAV/FAST-UAV/src/fastuav/notebooks/workdir/problem_inputs.xml'

In [14]:
INPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_inputs.xml")
oad.variable_viewer(INPUT_FILE)

In [20]:
# Definition of the DoE

# Inputs to be varied (stabilized ranges for MathWorks-scale quadcopter)
x_dict = {
    "models:weight:propulsion:multirotor:battery:mass:reference": [0.2, 1.0],  # [kg]
    "mission:sizing:main_route:hover:duration": [6, 24]  # [min]
}

# Outputs to observe
y_list = [
    "data:weight:mtow",
    "data:weight:propulsion:multirotor:battery:mass",
    "data:propulsion:multirotor:battery:energy",
    "data:performance:range:cruise",
    "data:performance:endurance:hover",
]

# Run DoE
# Use LHS to avoid a dense grid of strongly infeasible points
method = "lhs"
n_samples = 40

df = doe_fast(method, x_dict, y_list, CONFIGURATION_FILE, n_samples)

if "optim_failed" in df.columns:
    fail_ratio = float(df["optim_failed"].mean())
    print(f"Optimization failure ratio: {fail_ratio:.1%}")

c:\Users\user\.conda\envs\FastUAV_edit\lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\user\.conda\envs\FastUAV_edit\lib\site-packages\scipy\optimize\_slsqp_py.py:441: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds



2 out of 40 optimizations failed.
Optimization failure ratio: 5.0%


In [16]:
# Plot DoE with Plotly

# Minor calculations on the dataframe for better plotting
df = df.drop(df[df.optim_failed == 1.0].index)
df["data:propulsion:multirotor:battery:energy:density"] = df["data:propulsion:multirotor:battery:energy"] / 3.6 / df["data:weight:propulsion:multirotor:battery:mass"]
df["data:propulsion:multirotor:battery:energy:density"] = df["data:propulsion:multirotor:battery:energy:density"].astype(int)

# Set variables to plot
x_name = "data:propulsion:multirotor:battery:energy:density"
y_name = "mission:sizing:main_route:hover:duration"
z_name = "data:weight:mtow"

# Reshape data into a grid
x_vals = sorted(df[x_name].unique())
y_vals = sorted(df[y_name].unique())
z_grid = df.pivot(index=y_name, columns=x_name, values=z_name).values

# Create the contour plot    
fig = go.Figure()
fig.add_trace(go.Contour(
    x=x_vals, y=y_vals, z=z_grid, 
    connectgaps=True,
    line_smoothing=0.0,
    contours=dict(start=4.0, end=12, size=0.5, showlabels=True),
    colorbar=dict(
        title='Total UAV mass [kg]', # title here
        titleside='right',
        titlefont=dict(size=16),
        ticks='outside'
    ),
    colorscale='RdBu_r'
))

# Update layout
fig.update_layout(title=None, width=600, height=400, paper_bgcolor='rgba(0,0,0,0)',  plot_bgcolor='rgba(0,0,0,0)', font=dict(size=14),
                 margin=dict(l=10, r=10, t=0, b=0))
fig.update_xaxes(title='Battery energy density [Wh/kg]', ticks='outside', dtick=15, range=[130,260])
fig.update_yaxes(title='Hover endurance (min)', ticks='outside', range=[10,41])

# Display the plot
fig.show()

## Example 3 : Sobol analysis
The following example provides a more advanced example of what can be done using the `doe_fast` module. Ipywidget is used to set up an interactive interface to run a Sobol' analysis.

*For more details on this feature, please refer to the notebook [4_Uncertainty_Analysis](4_Uncertainty_analysis.ipynb).*

In [ ]:
from fastuav.utils.postprocessing.sensitivity_analysis.sensitivity_analysis import sobol_analysis

# Retrieve the UAV design
CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "doe_sobol_analysis.yaml")

# Prefer quadcopter reference design; fallback to DJI M600 if unavailable
REFERENCE_DESIGN_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_outputs_quadcopter_mdo.xml")
if not pth.exists(REFERENCE_DESIGN_FILE):
    REFERENCE_DESIGN_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_outputs_DJI_M600_mdo.xml")

print("Reference design file:", REFERENCE_DESIGN_FILE)

In [ ]:
# Display deterministic results
oad.variable_viewer(REFERENCE_DESIGN_FILE)

In [ ]:
# Carry out a sensitivity analysis on the UAV model
sobol_analysis(CONFIGURATION_FILE, REFERENCE_DESIGN_FILE)

![img](./img/uncertainty.gif)